# 🐦 Exercícios — Otimização por Enxame de Partículas (PSO)

**Disciplina:** Inteligência Artificial | **Nível:** Intermediário

> Implemente e experimente o PSO em funções de benchmark clássicas.


## 1. PSO Básico — Visualizando as Partículas

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

def funcao_objetivo(pos):
    """Função de Sphere — mínimo global em (0,0,...) = 0."""
    return np.sum(pos**2)

class PSO:
    def __init__(self, n_particulas=30, n_dim=2, limites=(-5,5),
                 w=0.7, c1=1.5, c2=1.5):
        self.n = n_particulas
        self.d = n_dim
        self.lim = limites
        self.w = w; self.c1 = c1; self.c2 = c2
        
        # Inicialização
        self.pos = np.random.uniform(*limites, (n_particulas, n_dim))
        self.vel = np.random.uniform(-1, 1, (n_particulas, n_dim))
        self.pbest = self.pos.copy()
        self.pbest_val = np.array([funcao_objetivo(p) for p in self.pos])
        
        gb_idx = np.argmin(self.pbest_val)
        self.gbest = self.pbest[gb_idx].copy()
        self.gbest_val = self.pbest_val[gb_idx]
        
        self.historico_gbest = [self.gbest_val]
    
    def passo(self):
        r1 = np.random.rand(self.n, self.d)
        r2 = np.random.rand(self.n, self.d)
        self.vel = (self.w * self.vel +
                    self.c1*r1*(self.pbest - self.pos) +
                    self.c2*r2*(self.gbest - self.pos))
        self.pos = np.clip(self.pos + self.vel, *self.lim)
        
        vals = np.array([funcao_objetivo(p) for p in self.pos])
        melhorou = vals < self.pbest_val
        self.pbest[melhorou] = self.pos[melhorou]
        self.pbest_val[melhorou] = vals[melhorou]
        
        gb_idx = np.argmin(self.pbest_val)
        if self.pbest_val[gb_idx] < self.gbest_val:
            self.gbest = self.pbest[gb_idx].copy()
            self.gbest_val = self.pbest_val[gb_idx]
        self.historico_gbest.append(self.gbest_val)
    
    def executar(self, iteracoes=100):
        for _ in range(iteracoes): self.passo()

# Executar PSO
pso = PSO(n_particulas=30)
pso.executar(100)

print(f"Melhor solução: {pso.gbest}")
print(f"Valor ótimo:    {pso.gbest_val:.8f}  (esperado: ~0)")

# Gráfico de convergência
plt.figure(figsize=(8,3))
plt.semilogy(pso.historico_gbest, 'b-')
plt.xlabel('Iteração'); plt.ylabel('Melhor Valor (log)'); plt.title('Convergência PSO — Função Sphere'); plt.grid(True); plt.show()


## 2. Visualizando o Enxame em Ação

In [ ]:
# Visualização do enxame em função Himmelblau
def himmelblau(x, y):
    """4 mínimos globais, todos com valor 0."""
    return (x**2 + y - 11)**2 + (x + y**2 - 7)**2

np.random.seed(42)
pso_hb = PSO(n_particulas=30, n_dim=2, limites=(-5,5), w=0.6, c1=2.0, c2=2.0)

# Substituir a função objetivo
def funcao_hb(pos): return himmelblau(pos[0], pos[1])
pso_hb.pbest_val = np.array([funcao_hb(p) for p in pso_hb.pos])
gb_idx = np.argmin(pso_hb.pbest_val)
pso_hb.gbest = pso_hb.pbest[gb_idx].copy()
pso_hb.gbest_val = pso_hb.pbest_val[gb_idx]

# Capturar estados para animação (estática)
capturas = []
for it in range(50):
    if it in [0,5,15,49]:
        capturas.append((it, pso_hb.pos.copy(), pso_hb.gbest.copy()))
    r1=np.random.rand(30,2); r2=np.random.rand(30,2)
    pso_hb.vel=(pso_hb.w*pso_hb.vel+pso_hb.c1*r1*(pso_hb.pbest-pso_hb.pos)+pso_hb.c2*r2*(pso_hb.gbest-pso_hb.pos))
    pso_hb.pos=np.clip(pso_hb.pos+pso_hb.vel,-5,5)
    vals=np.array([funcao_hb(p) for p in pso_hb.pos])
    melhorou=vals<pso_hb.pbest_val; pso_hb.pbest[melhorou]=pso_hb.pos[melhorou]; pso_hb.pbest_val[melhorou]=vals[melhorou]
    gb=np.argmin(pso_hb.pbest_val)
    if pso_hb.pbest_val[gb]<pso_hb.gbest_val: pso_hb.gbest=pso_hb.pbest[gb].copy(); pso_hb.gbest_val=pso_hb.pbest_val[gb]

x_=y_=np.linspace(-5,5,200)
X_,Y_=np.meshgrid(x_,y_); Z_=himmelblau(X_,Y_)

fig,axes=plt.subplots(1,4,figsize=(16,4))
for ax,(it,pos,gb) in zip(axes,capturas):
    ax.contourf(X_,Y_,np.log1p(Z_),levels=20,cmap='viridis',alpha=0.7)
    ax.scatter(pos[:,0],pos[:,1],c='white',s=20,zorder=3)
    ax.scatter([gb[0]],[gb[1]],c='red',s=100,marker='*',zorder=5)
    ax.set_title(f'Iteração {it}'); ax.set_xlim(-5,5); ax.set_ylim(-5,5)
plt.suptitle('PSO na Função Himmelblau'); plt.tight_layout(); plt.show()
print(f"Melhor encontrado: ({pso_hb.gbest[0]:.3f}, {pso_hb.gbest[1]:.3f}) = {pso_hb.gbest_val:.4f}")


### 📝 Exercício 1

Experimente com diferentes valores do **peso de inércia** w (0.3, 0.6, 0.9) e dos coeficientes c1, c2. Observe:
- w alto → mais exploração ou mais exploração?
- c1 alto (cognitivo) vs c2 alto (social): qual converge mais rápido?

In [ ]:
configs = [
    (0.3, 1.5, 1.5, 'w=0.3'),
    (0.6, 1.5, 1.5, 'w=0.6'),
    (0.9, 1.5, 1.5, 'w=0.9'),
    (0.6, 2.5, 0.5, 'c1>>c2 (cognitivo)'),
    (0.6, 0.5, 2.5, 'c2>>c1 (social)'),
]
plt.figure(figsize=(10,4))
for w, c1, c2, label in configs:
    np.random.seed(42)
    p = PSO(n_particulas=30, w=w, c1=c1, c2=c2)
    p.executar(100)
    plt.semilogy(p.historico_gbest, label=label)
plt.xlabel('Iteração'); plt.ylabel('Melhor Valor'); plt.title('Impacto dos Parâmetros do PSO')
plt.legend(); plt.grid(True); plt.show()


### 📝 Exercício Final

Adapte o PSO para otimizar a **função de Rastrigin** em 10 dimensões. Quantas iterações são necessárias para chegar perto de 0?

`f(x) = 10n + Σ[x²ᵢ - 10·cos(2π·xᵢ)]`

In [ ]:
def rastrigin_nd(pos):
    n = len(pos)
    return 10*n + np.sum(pos**2 - 10*np.cos(2*np.pi*pos))

# ✏️ Adapte o PSO (substitua funcao_objetivo):
class PSO_Rastrigin(PSO):
    def passo(self):
        # Override para usar rastrigin_nd
        r1=np.random.rand(self.n,self.d); r2=np.random.rand(self.n,self.d)
        self.vel=(self.w*self.vel+self.c1*r1*(self.pbest-self.pos)+self.c2*r2*(self.gbest-self.pos))
        self.pos=np.clip(self.pos+self.vel,*self.lim)
        vals=np.array([rastrigin_nd(p) for p in self.pos])
        melhorou=vals<self.pbest_val
        self.pbest[melhorou]=self.pos[melhorou]; self.pbest_val[melhorou]=vals[melhorou]
        gb=np.argmin(self.pbest_val)
        if self.pbest_val[gb]<self.gbest_val: self.gbest=self.pbest[gb].copy(); self.gbest_val=self.pbest_val[gb]
        self.historico_gbest.append(self.gbest_val)

np.random.seed(42)
pso_r = PSO_Rastrigin(n_particulas=50, n_dim=10, limites=(-5.12,5.12), w=0.7, c1=1.5, c2=1.5)
pso_r.pbest_val=np.array([rastrigin_nd(p) for p in pso_r.pos])
gb=np.argmin(pso_r.pbest_val); pso_r.gbest=pso_r.pbest[gb].copy(); pso_r.gbest_val=pso_r.pbest_val[gb]
pso_r.executar(500)
print(f"Rastrigin 10D — Melhor: {pso_r.gbest_val:.4f}  (ótimo global = 0)")
plt.figure(figsize=(8,3)); plt.semilogy(pso_r.historico_gbest,'m-'); plt.title('PSO — Rastrigin 10D'); plt.xlabel('Iteração'); plt.grid(True); plt.show()
